### Broadcast Variables & Accumulators in Spark DataFrame

#### PART 1 — Broadcast Variables

What is it?
A read-only lookup table sent once to every executor and **cached** there for the lifetime of the application. Without broadcast, Spark sends the variable with every single task — very inefficient.

![image_1775459194332.png](./image_1775459194332.png "image_1775459194332.png")

##### Example 1 — Country Code Lookup

Sample Data Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, DoubleType


data = [
    (1, "Alice",   "IN", "Electronics", 5000),
    (2, "Bob",     "US", "Clothing",    3000),
    (3, "Charlie", "IN", "Electronics", 7000),
    (4, "Diana",   "EU", "Clothing",    4000),
    (5, "Eve",     "US", "Electronics", 8000),
    (6, "Frank",   "IN", "Food",        1500),   # unknown category
    (7, "Grace",   "XX", "Electronics", 9000),   # unknown country
]
df = spark.createDataFrame(data, ["order_id", "name", "country", "category", "amount"])

In [0]:
# Lookup table — too large to hardcode in every task efficiently
country_map = {
    "IN": "India",
    "US": "United States",
    "EU": "European Union"
}

# Broadcast it once to all executors 
# broadcase is not available in serverless computing. hence directly accessing the map.
bc_country = spark.sparkContext.broadcast(country_map)

# Use inside a UDF — bc_country.value accesses the dict on executor
country_lookup = udf(
    lambda code: bc_country.value.get(code, "Unknown"),
    StringType()
)

df.withColumn("country_name", country_lookup(col("country"))).show()



##### Example 2 — Category Discount Lookup

In [0]:
disc_map = {
    "Electronics": 10,
    "Clothing": 5,
    "Food": 2
}

bc_disc = spark.sparkContext.broadcast(disc_map)

discount_lookup = udf(
    lambda cat: bc_disc.value.get(cat, 0), DoubleType()
)
df.withColumn("discount", df.amount * discount_lookup(col("category")) / 100) \
  .withColumn("Rate After Discount", col("amount") -  col("discount")).show()

##### Broadcast Join (without UDF)

In [0]:
from pyspark.sql.functions import broadcast
disc_df = spark.createDataFrame([
    ("Electronics", 5),
    ("Clothing", 10),
    ("Food", 2)],
    ["category", "discount_pct"])

df.join(broadcast(disc_df), "category", "left") \
  .withColumn("discount Amount", col("amount") * col("discount_pct")/100) \
  .withColumn("Rate After Discount", col("amount") -  col("discount Amount")).show()

 

#### PART 2 — Accumulators

What is it?
A **write-only** variable on executors that only the driver can read. Used to count events or aggregate metrics across tasks without collecting all data to the driver.

![image_1775469536451.png](./image_1775469536451.png "image_1775469536451.png")

##### Example 1 — Count Bad Records


In [0]:
# Define accumulators
unknown_country_count = spark.sparkContext.accumulator(0)
unknown_category_count = spark.sparkContext.accumulator(0)

known_countries  = {"IN", "US", "EU"}
known_categories = {"Electronics", "Clothing", "Food"}

def validate_and_tag(row):
    global unknown_country_count, unknown_category_count

    if row["country"] not in known_countries:
        unknown_country_count += 1         # executor writes

    if row["category"] not in known_categories:
        unknown_category_count += 1        # executor writes

# Use foreach (action) — guarantees exactly-once execution
df.foreach(validate_and_tag)

# Driver reads the final values
print(f"Unknown countries  : {unknown_country_count.value}")
print(f"Unknown categories : {unknown_category_count.value}")